In [1]:
%env AG_DATA=/content/ag_data
%env HF_HOME=/content/ag_data/hf
!mkdir -p /content/exp/scripts /content/exp/inputs /content/exp/golden $AG_DATA/ref $AG_DATA/golden $AG_DATA/hf
%cd /content/exp
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

env: AG_DATA=/content/ag_data
env: HF_HOME=/content/ag_data/hf
/content/exp
name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07


In [2]:
!pip install -q "git+https://github.com/google-deepmind/alphagenome_research.git"
!pip install -q -U "jax[cuda12]"
!python -c "import jax; print(jax.devices())"
import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 592.4/592.4 kB 49.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 144.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa
!ls -lh $AG_DATA/ref

total 3.1G
-rw-r--r-- 1 root root 3.1G Dec 18  2025 GRCh38.p13.genome.fa


In [7]:
!test -f $AG_DATA/ref/GRCh38.p13.genome.fa.fai || wget -q -O $AG_DATA/ref/GRCh38.p13.genome.fa.fai https://storage.googleapis.com/alphagenome/reference/gencode/hg38/GRCh38.p13.genome.fa.fai
!ls -lh $AG_DATA/ref

total 3.1G
-rw-r--r-- 1 root root 3.1G Dec 18  2025 GRCh38.p13.genome.fa
-rw-r--r-- 1 root root  22K Dec 18  2025 GRCh38.p13.genome.fa.fai


In [4]:
!python scripts/make_input.py
!cat inputs/cyp2b6_131k.json

python3: can't open file '/content/exp/scripts/make_input.py': [Errno 2] No such file or directory
cat: inputs/cyp2b6_131k.json: No such file or directory


In [5]:
!git -C /content/ag pull -q 2>/dev/null || git clone -q -b experiments-profiling https://github.com/Shivaritha22/alphagenome-pytorch /content/ag
%cd /content/ag/experiments
!git log -1 --oneline

/content/ag/experiments
a64f7ad (HEAD -> experiments-profiling, origin/experiments-profiling) experiments: golden check (make_input + golden notebook)


In [8]:
!python scripts/make_input.py
!cat inputs/cyp2b6_131k.json

chr19:40939304-41070376:. | N bases: 0 | -> inputs/cyp2b6_131k_onehot.npy
{
  "name": "cyp2b6_131k",
  "assembly": "GRCh38.p13",
  "fasta": "/content/ag_data/ref/GRCh38.p13.genome.fa",
  "chrom": "chr19",
  "start": 40939304,
  "end": 41070376,
  "length": 131072,
  "n_bases": 0,
  "sha256_seq": "35f4645276292bbe5d3c0c0ddc06d157dc544f38b5c1c691c004ea57150fbec8",
  "sha256_onehot": "74b53b1e94bf994719be05a531707d81bdf65d5f80afe3d97c554250b7f3e6e0",
  "created": "2026-10-16"
}

In [ ]:
!python scripts/jax_version.py